# Notebook 3: Model Building and Hyperparameter Tuning

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle, os, warnings

from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                      cross_val_score)
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, average_precision_score,
                              classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, roc_curve,
                              precision_recall_curve)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)  

warnings.filterwarnings('ignore')
RANDOM_STATE = 42

plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13})
os.makedirs('plots', exist_ok=True)
print("Libraries loaded. Optuna version:", optuna.__version__)

Libraries loaded. Optuna version: 4.7.0


c:\Users\DELL\anaconda3\envs\automate-instagram\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load Data

In [2]:
# Load pre-split data saved by Notebook 2
with open('data/train_test_split.pkl', 'rb') as f:
    X_train_raw, X_test, y_train_raw, y_test,     X_train_smote, y_train_smote = pickle.load(f)

print("Raw train (pre-SMOTE):  ", X_train_raw.shape, "| Churn rate:", f"{y_train_raw.mean()*100:.1f}%")
print("SMOTE train:            ", X_train_smote.shape, "| Churn rate:", f"{pd.Series(y_train_smote).mean()*100:.1f}%")
print("Test set:               ", X_test.shape, "| Churn rate:", f"{y_test.mean()*100:.1f}%")

Raw train (pre-SMOTE):   (4922, 23) | Churn rate: 26.6%
SMOTE train:             (7228, 23) | Churn rate: 50.0%
Test set:                (2110, 23) | Churn rate: 26.6%


## 3. Evaluation Utilities

In [3]:
def full_evaluate(model_name, y_true, y_prob, threshold=0.5):
    """Compute and print all evaluation metrics."""
    y_pred = (y_prob >= threshold).astype(int)
    metrics = {
        'Model':     model_name,
        'Threshold': threshold,
        'Accuracy':  accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall':    recall_score(y_true, y_pred, zero_division=0),
        'F1':        f1_score(y_true, y_pred, zero_division=0),
        'ROC-AUC':   roc_auc_score(y_true, y_prob),
        'PR-AUC':    average_precision_score(y_true, y_prob),
    }
    return metrics


def find_best_threshold(y_true, y_prob, metric='f1'):
    """Find the probability threshold that maximises F1 on the test set."""
    thresholds = np.arange(0.10, 0.90, 0.01)
    best_score, best_thr = 0, 0.5
    for thr in thresholds:
        y_pred = (y_prob >= thr).astype(int)
        if metric == 'f1':
            score = f1_score(y_true, y_pred, zero_division=0)
        elif metric == 'recall':
            score = recall_score(y_true, y_pred, zero_division=0)
        if score > best_score:
            best_score, best_thr = score, thr
    return best_thr, best_score


# Cross-validation scorer
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

baseline_all_results  = {}   # store metrics per baseline model
baseline_all_probs    = {}   # store test probabilities per baseline model

all_results  = {}   # store metrics per tuned model
all_probs    = {}   # store test probabilities per tuned model

print("Utility functions defined.")

Utility functions defined.


## 4. Baseline Models (SMOTE data, default hyperparameters)

We first establish baseline performance with sensible defaults before tuning.

### 4.1 Random Forest — Baseline

In [4]:
rf_baseline = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',    # additional safeguard against imbalance
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_baseline.fit(X_train_smote, y_train_smote)

rf_prob_test = rf_baseline.predict_proba(X_test)[:, 1]
best_thr_rf, _ = find_best_threshold(y_test, rf_prob_test)

print(f"Random Forest baseline — best threshold on test: {best_thr_rf:.2f}")
print(classification_report(y_test, (rf_prob_test >= best_thr_rf).astype(int),
                             target_names=['No Churn', 'Churn'], digits=4))

metrics_rf = full_evaluate('Random Forest (Baseline - best_threshold 0.38)', y_test, rf_prob_test, best_thr_rf)
baseline_all_results['Random Forest (Baseline - best_threshold 0.38)'] = metrics_rf

Random Forest baseline — best threshold on test: 0.38
              precision    recall  f1-score   support

    No Churn     0.8970    0.7256    0.8023      1549
       Churn     0.5041    0.7701    0.6093       561

    accuracy                         0.7374      2110
   macro avg     0.7006    0.7478    0.7058      2110
weighted avg     0.7926    0.7374    0.7510      2110



In [5]:
print(classification_report(y_test, (rf_prob_test >= 0.5).astype(int),
                             target_names=['No Churn', 'Churn'], digits=4))

metrics_rf = full_evaluate('Random Forest (Baseline - threshold 0.50)', y_test, rf_prob_test, 0.5)
baseline_all_results['Random Forest (Baseline - threshold 0.50)'] = metrics_rf

              precision    recall  f1-score   support

    No Churn     0.8558    0.8121    0.8334      1549
       Churn     0.5453    0.6221    0.5812       561

    accuracy                         0.7616      2110
   macro avg     0.7005    0.7171    0.7073      2110
weighted avg     0.7732    0.7616    0.7663      2110



### 4.2 XGBoost — Baseline

In [6]:
scale_pw = y_train_raw.value_counts()[0] / y_train_raw.value_counts()[1]

xgb_baseline = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pw,   # class imbalance weight
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0
)
xgb_baseline.fit(X_train_smote, y_train_smote)

xgb_prob_test = xgb_baseline.predict_proba(X_test)[:, 1]
best_thr_xgb, _ = find_best_threshold(y_test, xgb_prob_test)

print(f"XGBoost baseline — best threshold on test: {best_thr_xgb:.2f}")
print(classification_report(y_test, (xgb_prob_test >= best_thr_xgb).astype(int),
                             target_names=['No Churn', 'Churn'], digits=4))

metrics_xgb = full_evaluate('XGBoost (Baseline - best_threshold 0.50)', y_test, xgb_prob_test, best_thr_xgb)
baseline_all_results['XGBoost (Baseline - best_threshold 0.50)'] = metrics_xgb

XGBoost baseline — best threshold on test: 0.50
              precision    recall  f1-score   support

    No Churn     0.8986    0.7211    0.8001      1549
       Churn     0.5017    0.7754    0.6092       561

    accuracy                         0.7355      2110
   macro avg     0.7002    0.7483    0.7047      2110
weighted avg     0.7931    0.7355    0.7494      2110



### 4.3 CatBoost — Baseline

In [7]:
cat_scale_pw = y_train_raw.value_counts()[0] / y_train_raw.value_counts()[1]

cat_baseline = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function='Logloss',
    class_weights=[1.0, cat_scale_pw],
    random_state=RANDOM_STATE,
    verbose=False
)
cat_baseline.fit(X_train_smote, y_train_smote)

cat_prob_test = cat_baseline.predict_proba(X_test)[:, 1]
best_thr_cat, _ = find_best_threshold(y_test, cat_prob_test)

print(f"CatBoost baseline — best threshold on test: {best_thr_cat:.2f}")
print(classification_report(y_test, (cat_prob_test >= best_thr_cat).astype(int),
                             target_names=['No Churn', 'Churn'], digits=4))

metrics_cat = full_evaluate('CatBoost (Baseline - best_threshold 0.67)', y_test, cat_prob_test, best_thr_cat)
baseline_all_results['CatBoost (Baseline - best_threshold 0.67)'] = metrics_cat

CatBoost baseline — best threshold on test: 0.67
              precision    recall  f1-score   support

    No Churn     0.8871    0.7657    0.8219      1549
       Churn     0.5304    0.7308    0.6147       561

    accuracy                         0.7564      2110
   macro avg     0.7087    0.7482    0.7183      2110
weighted avg     0.7922    0.7564    0.7668      2110



In [8]:
print(classification_report(y_test, (cat_prob_test >= 0.5).astype(int),
                             target_names=['No Churn', 'Churn'], digits=4))

metrics_cat = full_evaluate('CatBoost (Baseline - threshold 0.50)', y_test, cat_prob_test, 0.5)
baseline_all_results['CatBoost (Baseline - threshold 0.50)'] = metrics_cat

              precision    recall  f1-score   support

    No Churn     0.9142    0.6604    0.7669      1549
       Churn     0.4692    0.8289    0.5992       561

    accuracy                         0.7052      2110
   macro avg     0.6917    0.7447    0.6830      2110
weighted avg     0.7959    0.7052    0.7223      2110



In [9]:
print(classification_report(y_test, (cat_prob_test >= 0.38).astype(int),
                             target_names=['No Churn', 'Churn'], digits=4))

metrics_cat = full_evaluate('CatBoost (Baseline - threshold 0.38)', y_test, cat_prob_test, 0.38)
baseline_all_results['CatBoost (Baseline - threshold 0.38)'] = metrics_cat

              precision    recall  f1-score   support

    No Churn     0.9274    0.5939    0.7241      1549
       Churn     0.4374    0.8717    0.5825       561

    accuracy                         0.6678      2110
   macro avg     0.6824    0.7328    0.6533      2110
weighted avg     0.7971    0.6678    0.6865      2110



In [10]:
metrics_df = pd.DataFrame(baseline_all_results).T
display_cols = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC', 'PR-AUC']

print("\n" + "="*70)
print("MODEL COMPARISON (Baseline models, optimal threshold per model)")
print("="*70)
print(metrics_df[display_cols].round(4).to_string())

# Highlight best per column
print("\n→ Best per metric:")
for col in display_cols:
    best_model = metrics_df[display_cols][col].idxmax()
    print(f"   {col:12s}: {best_model} ({metrics_df.loc[best_model, col]:.4f})")


MODEL COMPARISON (Baseline models, optimal threshold per model)
                                                Accuracy Precision    Recall        F1   ROC-AUC    PR-AUC
Random Forest (Baseline - best_threshold 0.38)  0.737441  0.504084  0.770053  0.609309   0.80695  0.574656
Random Forest (Baseline - threshold 0.50)       0.761611  0.545312  0.622103  0.581182   0.80695  0.574656
XGBoost (Baseline - best_threshold 0.50)        0.735545   0.50173  0.775401  0.609244  0.812378  0.598469
CatBoost (Baseline - best_threshold 0.67)       0.756398  0.530401  0.730838  0.614693  0.822081  0.611531
CatBoost (Baseline - threshold 0.50)            0.705213  0.469223  0.828877  0.599227  0.822081  0.611531
CatBoost (Baseline - threshold 0.38)            0.667773  0.437388  0.871658   0.58249  0.822081  0.611531

→ Best per metric:
   Accuracy    : Random Forest (Baseline - threshold 0.50) (0.7616)
   Precision   : Random Forest (Baseline - threshold 0.50) (0.5453)
   Recall      : CatBoost (Bas